## Setup

In [4]:
master = "spark://spark-master:7077"  # use "spark://spark-master:7077" when running inside the Docker cluster
docker_data_path = "../data/"

dataFolder = docker_data_path if master.startswith("spark://") else "../data/"

In [5]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master(master) \
    .appName("ParComp_Project") \
    .getOrCreate()
    # .config("spark.ui.showConsoleProgress", "false.set
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
spark.conf.set("spark.sql.adaptive.enabled", "false")

print("Spark UI:", spark.sparkContext.uiWebUrl)
print("Default parallelism (cores seen by the driver):", spark.sparkContext.defaultParallelism)


Spark UI: http://b2bda2fd422d:4040
Default parallelism (cores seen by the driver): 2


In [6]:
print(spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))
print(spark.conf.get("spark.sql.adaptive.enabled"))

-1
false


In [7]:
import sys
sys.path.append("../src")
from spark_profiler import SparkProfiler

profiler = SparkProfiler(spark)


In [8]:
with profiler.profile("0-smoke-test"):
   
    taxi_df = spark.read.parquet(dataFolder + "tripdata_2026/yellow_tripdata_2026-01.parquet")
    
    print("Partitions right after read:", taxi_df.rdd.getNumPartitions())
    print("Total columns:", len(taxi_df.columns))
    
    
    print(taxi_df.groupBy("PULocationID").count().orderBy("PULocationID").limit(10).collect())

Partitions right after read: 2
Total columns: 20


[Row(PULocationID=1, count=486), Row(PULocationID=2, count=3), Row(PULocationID=3, count=491), Row(PULocationID=4, count=16613), Row(PULocationID=5, count=4), Row(PULocationID=6, count=95), Row(PULocationID=7, count=10044), Row(PULocationID=8, count=56), Row(PULocationID=9, count=198), Row(PULocationID=10, count=2050)]
[0-smoke-test] wall time: 18.77s (3 stages, 203 tasks) - see profiler.report() / profiler.compare(...)


## Baseline

# M1 - Data ingestion and cleaning. 
Load the monthly Parquet files. Enforce schema (cast types, handle nulls). Filter out invalid records on all of the following criteria

### Step 1: Read data

We can read different variations of datasets(Whole data, single year or specific month)

In [9]:
from pathlib import Path
from typing import Literal

parquet_files: list[Path] = []
specific_files: Literal["directory", ""] = "directory"
year_dir: Literal["/tripdata_2025", "/tripdata_2026"] = "/tripdata_2025"
spec_files: list[str] = [] # ["yellow_tripdata_2026-01.parquet"]

parquet_files: list[Path] = []

if spec_files:
    parquet_files.extend(sorted(Path(dataFolder).rglob(*spec_files)))
elif not specific_files:
    parquet_files.extend(sorted(Path(dataFolder).rglob("*.parquet")))
else:
    parquet_files.extend(sorted(Path(dataFolder + year_dir).rglob("*.parquet")))

print(parquet_files)

[PosixPath('../data/tripdata_2025/yellow_tripdata_2025-01.parquet'), PosixPath('../data/tripdata_2025/yellow_tripdata_2025-02.parquet'), PosixPath('../data/tripdata_2025/yellow_tripdata_2025-03.parquet'), PosixPath('../data/tripdata_2025/yellow_tripdata_2025-04.parquet'), PosixPath('../data/tripdata_2025/yellow_tripdata_2025-05.parquet'), PosixPath('../data/tripdata_2025/yellow_tripdata_2025-06.parquet'), PosixPath('../data/tripdata_2025/yellow_tripdata_2025-07.parquet'), PosixPath('../data/tripdata_2025/yellow_tripdata_2025-08.parquet'), PosixPath('../data/tripdata_2025/yellow_tripdata_2025-09.parquet'), PosixPath('../data/tripdata_2025/yellow_tripdata_2025-10.parquet'), PosixPath('../data/tripdata_2025/yellow_tripdata_2025-11.parquet'), PosixPath('../data/tripdata_2025/yellow_tripdata_2025-12.parquet')]


### Step 2: Enforce the schema
Because we don't work with inferSchema based on performance reasons (one job more for inferSchema), we have to enforce the datatypes of Columns in another way. 

In [10]:
from pyspark.sql import functions as F 
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    LongType,
    TimestampType,
    StructField,
    StructType,
)

schema = StructType([
    StructField("tpep_pickup_datetime", TimestampType(), True),
    StructField("tpep_dropoff_datetime", TimestampType(), True),
    StructField("PULocationID", IntegerType(), True),
    StructField("DOLocationID", IntegerType(), True),
    StructField("passenger_count", LongType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", LongType(), True),
    StructField("payment_type", LongType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
])
with profiler.profile("task-1_read_data"):
    raw_trips = spark.read.schema(schema)\
        .parquet(*[str(p) for p in parquet_files]) \
        .select(
            "tpep_pickup_datetime",
            "tpep_dropoff_datetime",
            "PULocationID",
            "DOLocationID",
            F.column("passenger_count").cast(IntegerType()),
            "trip_distance",
            F.column("RatecodeID").cast(IntegerType()),
            F.column("payment_type").cast(IntegerType()),
            "fare_amount",
            "tip_amount",
            "total_amount"
        )
    print(raw_trips.limit(5).toPandas())

/opt/spark/python/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


  tpep_pickup_datetime tpep_dropoff_datetime  PULocationID  DOLocationID  \
0  2025-05-01 00:07:06   2025-05-01 00:24:15           140           202   
1  2025-05-01 00:07:44   2025-05-01 00:14:27           234           161   
2  2025-05-01 00:15:56   2025-05-01 00:23:53           161           234   
3  2025-05-01 00:00:09   2025-05-01 00:25:29           138            90   
4  2025-05-01 00:45:07   2025-05-01 00:52:45            90           231   

   passenger_count  trip_distance  RatecodeID  payment_type  fare_amount  \
0                1           3.70           1             1         18.4   
1                1           1.03           1             1          8.6   
2                1           1.57           1             2         10.0   
3                1           9.48           1             1         40.8   
4                1           1.80           1             1         10.0   

   tip_amount  total_amount  
0        4.85         29.00  
1        4.30         18.6

### Step 3: Handle invalid Inputs

Handle all null and invalid values and print them.

In [36]:
invalid_fare      = (F.col("fare_amount").isNull()) | (F.col("fare_amount") <= 0)
invalid_distance  = (F.col("trip_distance").isNull()) | (F.col("trip_distance") <= 0)
invalid_passengers= (F.col("passenger_count").isNull()) | (F.col("passenger_count") == 0) | (F.col("passenger_count") > 6)
invalid_timestamps= (F.col("tpep_pickup_datetime").isNull()) \
                    | (F.col("tpep_dropoff_datetime").isNull()) \
                    | (F.col("tpep_dropoff_datetime") <= F.col("tpep_pickup_datetime"))
invalid_locations = (F.col("PULocationID").isNull()) | (F.col("DOLocationID").isNull()) | \
                    (~F.col("PULocationID").between(1, 263)) | (~F.col("DOLocationID").between(1, 263))

invalid_any = invalid_fare | invalid_distance | invalid_passengers | invalid_timestamps | invalid_locations

trips = raw_trips.withColumn("year", F.year("tpep_pickup_datetime"))

with profiler.profile("task-1_clean_data"):
    cleaning_stats = trips \
        .groupBy("year") \
        .agg(
            F.count("*").alias("total_records_processed"),
            F.sum(invalid_fare.cast(IntegerType())).alias("removed_invalid_fare"), 
            F.sum(invalid_distance.cast(IntegerType())).alias("removed_invalid_distance"),
            F.sum(invalid_passengers.cast(IntegerType())).alias("removed_invalid_passengers"),
            F.sum(invalid_timestamps.cast(IntegerType())).alias("removed_invalid_timestamps"),
            F.sum(invalid_locations.cast(IntegerType())).alias("removed_invalid_locations"),
            F.sum(invalid_any.cast(IntegerType())).alias("total_records_removed")
        ) \
        .orderBy("year")
    clean_trips = trips.filter(~invalid_any)
    # trips.cache()
    trips.unpersist()
    print("--- M1 Data Cleaning Statistics Report ---")
    cleaning_stats.show(truncate=False)

--- M1 Data Cleaning Statistics Report ---


+----+-----------------------+--------------------+------------------------+--------------------------+--------------------------+-------------------------+---------------------+
|year|total_records_processed|removed_invalid_fare|removed_invalid_distance|removed_invalid_passengers|removed_invalid_timestamps|removed_invalid_locations|total_records_removed|
+----+-----------------------+--------------------+------------------------+--------------------------+--------------------------+-------------------------+---------------------+
|2007|1                      |0                   |0                       |0                         |0                         |0                        |0                    |
|2008|1                      |0                   |0                       |0                         |0                         |0                        |0                    |
|2009|6                      |0                   |0                       |0                         |0 

In [12]:
profiler.report()

,wall_time_s,num_jobs,num_stages,num_tasks,executor_run_time_ms,executor_cpu_time_ms,input_bytes,output_bytes,shuffle_read_bytes,shuffle_write_bytes,memory_spilled_bytes,disk_spilled_bytes
run,,,,,,,,,,,,
0-smoke-test,18.77,2,3,203,10614,7103,21.9KB,0.0B,21.3KB,21.3KB,0.0B,0.0B
task-1_read_data,1.29,1,1,1,379,219,13.2KB,0.0B,0.0B,0.0B,0.0B,0.0B
task-1_clean_data,17.77,1,2,210,28349,20629,142.9KB,0.0B,1.4KB,1.4KB,0.0B,0.0B


In [13]:
trips.explain(True)

== Parsed Logical Plan ==
'Project [unresolvedstarwithcolumns(year, 'year('tpep_pickup_datetime), None)]
+- Project [tpep_pickup_datetime#45, tpep_dropoff_datetime#46, PULocationID#47, DOLocationID#48, cast(passenger_count#49L as int) AS passenger_count#56, trip_distance#50, cast(RatecodeID#51L as int) AS RatecodeID#57, cast(payment_type#52L as int) AS payment_type#58, fare_amount#53, tip_amount#54, total_amount#55]
   +- Relation [tpep_pickup_datetime#45,tpep_dropoff_datetime#46,PULocationID#47,DOLocationID#48,passenger_count#49L,trip_distance#50,RatecodeID#51L,payment_type#52L,fare_amount#53,tip_amount#54,total_amount#55] parquet

== Analyzed Logical Plan ==
tpep_pickup_datetime: timestamp, tpep_dropoff_datetime: timestamp, PULocationID: int, DOLocationID: int, passenger_count: int, trip_distance: double, RatecodeID: int, payment_type: int, fare_amount: double, tip_amount: double, total_amount: double, year: int
Project [tpep_pickup_datetime#45, tpep_dropoff_datetime#46, PULocationID

# M2 - Temporal demand analysis

In [30]:
zone_lookup_path = dataFolder + "taxi_zone_lookup.csv"

zones = spark.read.csv(zone_lookup_path, header=True) \
    .select(
        F.col("LocationID").cast(IntegerType()).alias("LocationID"),
        F.col("Borough").alias("pickup_borough")
    )

trips = clean_trips \
    .withColumn("day_of_week", F.dayofweek("tpep_pickup_datetime")) \
    .withColumn("hour_of_day", F.hour("tpep_pickup_datetime")) \
    .withColumn("trip_duration", 
        (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60
    )


trips = trips.join(
    F.broadcast(zones),
    on=trips.PULocationID == zones.LocationID,
    how="left")

demand = trips \
    .groupBy("pickup_borough", "day_of_week", "hour_of_day") \
    .agg(
        F.count("*").alias("total_trip_count"),
        F.avg("fare_amount").alias("mean_fare"),
        F.avg("trip_duration").alias("mean_trip_duration"),
        F.avg("trip_distance").alias("mean_trip_distance")
    )
    # .orderBy("pickup_borough", "day_of_week", "hour_of_day")

output_path = "../output"

with profiler.profile("task-2_temporal_demand_analysis"):
    demand.write \
        .partitionBy("pickup_borough") \
        .parquet(f"{output_path}/demand_by_borough", mode="overwrite")

with profiler.profile("task-2_show_10_lines"):
    print("--- M2 Temporal Demand Analysis 10 first ---")
    demand.show(10, truncate=False)

26/11/06 17:08:14 WARN TaskSetManager: Lost task 1.0 in stage 84.0 (TID 8566) (172.20.0.4 executor 0): org.apache.spark.memory.SparkOutOfMemoryError: [AGGREGATE_OUT_OF_MEMORY] No enough memory for aggregation SQLSTATE: 82001
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage2.hashAgg_doConsume_0$(Unknown Source)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage2.hashAgg_doAggregateWithKeys_0$(Unknown Source)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage2.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:44)
	at org.apache.spark.sql.execution.WholeStageCodegenEvaluatorFactory$WholeStageCodegenPartitionEvaluator$$anon$1.hasNext(WholeStageCodegenEvaluatorFactory.scala:50)
	at scala.collection.Iterator$$anon$9.hasNext(Iterator.scala:593)
	at org.apache.spark.shuffle.sort.BypassMergeSortShuffl

[task-2_temporal_demand_analysis] wall time: 189.79s (3 stages, 211 tasks) - see profiler.report() / profiler.compare(...)
--- M2 Temporal Demand Analysis 10 first ---


26/11/06 17:11:11 WARN TaskSetManager: Lost task 1.0 in stage 87.0 (TID 8778) (172.20.0.4 executor 0): org.apache.spark.memory.SparkOutOfMemoryError: [AGGREGATE_OUT_OF_MEMORY] No enough memory for aggregation SQLSTATE: 82001
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage2.hashAgg_doConsume_0$(Unknown Source)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage2.hashAgg_doAggregateWithKeys_0$(Unknown Source)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage2.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:44)
	at org.apache.spark.sql.execution.WholeStageCodegenEvaluatorFactory$WholeStageCodegenPartitionEvaluator$$anon$1.hasNext(WholeStageCodegenEvaluatorFactory.scala:50)
	at scala.collection.Iterator$$anon$9.hasNext(Iterator.scala:593)
	at org.apache.spark.shuffle.sort.BypassMergeSortShuffl

+--------------+-----------+-----------+----------------+------------------+------------------+------------------+
|pickup_borough|day_of_week|hour_of_day|total_trip_count|mean_fare         |mean_trip_duration|mean_trip_distance|
+--------------+-----------+-----------+----------------+------------------+------------------+------------------+
|EWR           |3          |5          |4               |58.75             |0.1               |0.015             |
|Staten Island |1          |8          |1               |75.0              |23.4              |14.88             |
|Bronx         |1          |10         |1170            |35.32070940170941 |39.09465811965813 |9.177991452991453 |
|Manhattan     |1          |15         |232206          |16.294117550795352|15.148674237530463|2.501701721747068 |
|Bronx         |4          |2          |29              |24.824137931034485|15.158045977011495|4.0937931034482755|
|EWR           |7          |12         |4               |125.2475          |1.79

In [15]:
trips.explain(True)

== Parsed Logical Plan ==
Join LeftOuter, (PULocationID#47 = LocationID#153)
:- Project [tpep_pickup_datetime#45, tpep_dropoff_datetime#46, PULocationID#47, DOLocationID#48, passenger_count#56, trip_distance#50, RatecodeID#57, payment_type#58, fare_amount#53, tip_amount#54, total_amount#55, year#60, day_of_week#156, hour_of_day#157, (cast((unix_timestamp(tpep_dropoff_datetime#46, yyyy-MM-dd HH:mm:ss, Some(Etc/UTC), true) - unix_timestamp(tpep_pickup_datetime#45, yyyy-MM-dd HH:mm:ss, Some(Etc/UTC), true)) as double) / cast(60 as double)) AS trip_duration#158]
:  +- Project [tpep_pickup_datetime#45, tpep_dropoff_datetime#46, PULocationID#47, DOLocationID#48, passenger_count#56, trip_distance#50, RatecodeID#57, payment_type#58, fare_amount#53, tip_amount#54, total_amount#55, year#60, day_of_week#156, hour(tpep_pickup_datetime#45, Some(Etc/UTC)) AS hour_of_day#157]
:     +- Project [tpep_pickup_datetime#45, tpep_dropoff_datetime#46, PULocationID#47, DOLocationID#48, passenger_count#56, tri

# M3 - Zone-level statistics

In [38]:
from pyspark.sql.window import Window
from pyspark.sql.types import DateType
zones = spark.read.csv(zone_lookup_path, header=True) \
    .select(
        F.col("LocationID").cast(IntegerType()).alias("PULocationID"),
        F.col("Borough").alias("pickup_borough"),
        F.col("Zone").alias("pickup_zone")
    )
# FIX: first calculate zone table and after that join with broadcast join

# old version
# monthly = clean_trips \
#     .join(zones, on="PULocationID", how="left") \
#     .withColumn("month", F.date_trunc("month", "tpep_pickup_datetime").cast(DateType())) \
#     .withColumn("month_idx", F.year("month") * 12 + F.month("month")) \
#     .groupBy("pickup_borough", "PULocationID", "pickup_zone", "month", "month_idx") \
#     .agg(
#         F.count("*").alias("trip_count"),
#         F.avg("fare_amount").alias("mean_fare"),
#         F.std("fare_amount").alias("std_fare"),
#         F.avg("trip_distance").alias("mean_trip_distance"),
#         F.std("trip_distance").alias("std_trip_distance")
#     )

# new optimized version
monthly_trips = clean_trips \
    .withColumn("month", F.date_trunc("month", "tpep_pickup_datetime").cast(DateType())) \
    .withColumn("month_idx", F.year("month") * 12 + F.month("month")) \
    .groupBy("PULocationID", "month", "month_idx") \
    .agg(
        F.count("*").alias("trip_count"),
        F.avg("fare_amount").alias("mean_fare"),
        F.std("fare_amount").alias("std_fare"),
        F.avg("trip_distance").alias("mean_trip_distance"),
        F.std("trip_distance").alias("std_trip_distance")
    )

monthly = monthly_trips \
    .join(
        F.broadcast(zones),
        on="PULocationID",
        how="left"
    )

window_rank = Window.partitionBy("pickup_borough", "month").orderBy(F.desc("trip_count"))
result = monthly.withColumn("volume_rank", F.rank().over(window_rank))

window_zone_3months = Window.partitionBy("PULocationID") \
    .orderBy("month_idx") \
    .rangeBetween(-2, 0)

window_borough_3months = Window.partitionBy("pickup_borough") \
    .orderBy("month_idx") \
    .rangeBetween(-2, 0)

result = result \
    .withColumn("zone_trips_3m", F.sum("trip_count").over(window_zone_3months)) \
    .withColumn("borough_trips_3m", F.sum("trip_count").over(window_borough_3months)) \
    .withColumn("share_3m", F.col("zone_trips_3m") / F.col("borough_trips_3m"))

w_hist = (Window.partitionBy("PULocationID").orderBy("month_idx")
          .rangeBetween(-3, -1))

result = result \
    .withColumn("roll_mean", F.avg("trip_count").over(w_hist)) \
    .withColumn("roll_std",  F.stddev("trip_count").over(w_hist)) \
    .withColumn(
        "is_anomaly", 
        F.coalesce(
            F.abs(F.col("trip_count") - F.col("roll_mean")) > 1.5 * F.col("roll_std"),
            F.lit(False)
        )
    )

final = result.orderBy("pickup_borough", "month", "volume_rank")

# with profiler.profile("task-3_zone_level_statistics_before"):
#     final.write.mode("overwrite").partitionBy("pickup_borough") \
#         .parquet("/project/output/zone_monthly_stats")
    
with profiler.profile("task-3_zone_level_statistics_after"):
    final.write.mode("overwrite").partitionBy("pickup_borough") \
        .parquet("/project/output/zone_monthly_stats")

final.filter("is_anomaly").show(20, truncate=False)

[task-3_zone_level_statistics_after] wall time: 127.59s (14 stages, 2221 tasks) - see profiler.report() / profiler.compare(...)


+------------+----------+---------+----------+------------------+------------------+------------------+------------------+--------------+---------------------------------+-----------+-------------+----------------+--------------------+---------+------------------+----------+
|PULocationID|month     |month_idx|trip_count|mean_fare         |std_fare          |mean_trip_distance|std_trip_distance |pickup_borough|pickup_zone                      |volume_rank|zone_trips_3m|borough_trips_3m|share_3m            |roll_mean|roll_std          |is_anomaly|
+------------+----------+---------+----------+------------------+------------------+------------------+------------------+--------------+---------------------------------+-----------+-------------+----------------+--------------------+---------+------------------+----------+
|168         |2025-03-01|24303    |377       |27.08649867374004 |13.947340894495229|5.604801061007963 |4.572972405618915 |Bronx         |Mott Haven/Port Morris           |1

In [39]:
profiler.compare("task-3_zone_level_statistics_before", "task-3_zone_level_statistics_after")

,task-3_zone_level_statistics_before,task-3_zone_level_statistics_after,change
metric,,,
wall_time_s,157.19,127.59,-18.8%
num_jobs,2,3,+50.0%
num_stages,15,14,-6.7%
num_tasks,2222,2221,-0.0%
executor_run_time_ms,139055,78127,-43.8%
executor_cpu_time_ms,96682,39209,-59.4%
input_bytes,10.7MB,10.7MB,+0.0%
output_bytes,1.1MB,1.1MB,-0.0%
shuffle_read_bytes,527.5MB,2.3MB,-99.6%


In [40]:
print("--- M3 Zone-Level Window Statistics (First 10 Rows) ---")
final.show(3, vertical=True, truncate=False)

--- M3 Zone-Level Window Statistics (First 10 Rows) ---


-RECORD 0----------------------------------------------
 PULocationID       | 69                               
 month              | 2025-01-01                       
 month_idx          | 24301                            
 trip_count         | 362                              
 mean_fare          | 32.347790055248616               
 std_fare           | 14.49599268649379                
 mean_trip_distance | 7.842458563535917                
 std_trip_distance  | 5.808109840740017                
 pickup_borough     | Bronx                            
 pickup_zone        | East Concourse/Concourse Village 
 volume_rank        | 1                                
 zone_trips_3m      | 362                              
 borough_trips_3m   | 5876                             
 share_3m           | 0.06160653505786249              
 roll_mean          | NULL                             
 roll_std           | NULL                             
 is_anomaly         | false                     

In [41]:
final.explain(True)

== Parsed Logical Plan ==
'Sort ['pickup_borough ASC NULLS FIRST, 'month ASC NULLS FIRST, 'volume_rank ASC NULLS FIRST], true
+- Project [PULocationID#47, month#5755, month_idx#5756, trip_count#5757L, mean_fare#5758, std_fare#5759, mean_trip_distance#5760, std_trip_distance#5761, pickup_borough#5752, pickup_zone#5753, volume_rank#5813, zone_trips_3m#5822L, borough_trips_3m#5824L, share_3m#5826, roll_mean#5827, roll_std#5829, coalesce((abs((cast(trip_count#5757L as double) - roll_mean#5827)) > (roll_std#5829 * 1.5)), false) AS is_anomaly#5856]
   +- Project [PULocationID#47, month#5755, month_idx#5756, trip_count#5757L, mean_fare#5758, std_fare#5759, mean_trip_distance#5760, std_trip_distance#5761, pickup_borough#5752, pickup_zone#5753, volume_rank#5813, zone_trips_3m#5822L, borough_trips_3m#5824L, share_3m#5826, roll_mean#5827, roll_std#5829]
      +- Project [PULocationID#47, month#5755, month_idx#5756, trip_count#5757L, mean_fare#5758, std_fare#5759, mean_trip_distance#5760, std_trip

# O1

In [ ]:
spark.sparkContext.setJobGroup("O1_ANOMALY_BASELINE", "O1: Fare Anomaly Detection Baseline")

with profiler.profile("O1-fare-anomaly-baseline"):
    
    o1_base = (clean_trips
        .filter(F.col("trip_distance") > 0)
        .withColumn("fare_per_dist", F.col("fare_amount") / F.col("trip_distance"))
        .withColumn("hour_of_day", F.hour(F.col("tpep_pickup_datetime")))
        .withColumn("is_weekend", F.dayofweek(F.col("tpep_pickup_datetime")).isin([1, 7]))
    )

    o1_stats = (o1_base
        .groupBy("PULocationID", "hour_of_day")
        .agg(
            F.mean("fare_per_dist").alias("mean_fare_per_dist"),
            F.std("fare_per_dist").alias("std_fare_per_dist")
        )
    )


    o1_flagged = (o1_base
        .join(o1_stats, on=["PULocationID", "hour_of_day"], how="inner")
        .join(
            zones.select(
                F.col("PULocationID"),
                "pickup_borough"
            ),
            on="PULocationID",
            how="inner"
        )
        .withColumn(
            "is_fare_anomaly",
            F.when(
                F.col("std_fare_per_dist").isNotNull() & (F.col("std_fare_per_dist") > 0),
                F.col("fare_per_dist") > (F.col("mean_fare_per_dist") + (3 * F.col("std_fare_per_dist"))) 
            ).otherwise(False)
        )
    )

    o1_summary = (o1_flagged
        .groupBy("pickup_borough", "payment_type", "is_weekend")
        .agg(
            F.count("*").alias("total_trips"),
            F.sum(F.when(F.col("is_fare_anomaly"), 1).otherwise(0)).alias("anomaly_trips"), # True if anomaly
            (F.sum(F.when(F.col("is_fare_anomaly"), 1).otherwise(0)) / F.count("*")).alias("anomaly_rate") 
        ) 
        .orderBy("pickup_borough", "payment_type", "is_weekend")
    )

    print("--- O1 Fare Anomaly Summary (First 10 Rows) ---")
    o1_summary.show(10, truncate=False)

spark.sparkContext.setJobGroup("", "")

#for optimizing
#1 broadcastjoin
#2 getting rid of 1st join completely

--- O1 Fare Anomaly Summary (First 10 Rows) ---


+--------------+------------+----------+-----------+-------------+--------------------+
|pickup_borough|payment_type|is_weekend|total_trips|anomaly_trips|anomaly_rate        |
+--------------+------------+----------+-----------+-------------+--------------------+
|Bronx         |1           |false     |101567     |1127         |0.011096123740978861|
|Bronx         |1           |true      |23029      |277          |0.0120283121281862  |
|Bronx         |2           |false     |4097       |118          |0.02880156211862338 |
|Bronx         |2           |true      |1826       |44           |0.024096385542168676|
|Bronx         |3           |false     |257        |39           |0.1517509727626459  |
|Bronx         |3           |true      |103        |6            |0.05825242718446602 |
|Bronx         |4           |false     |463        |40           |0.08639308855291576 |
|Bronx         |4           |true      |241        |23           |0.0954356846473029  |
|Brooklyn      |1           |fal

# O2

In [20]:
spark.sparkContext.setJobGroup("O2_FLOWS_BASELINE", "O2: Origin-Destination Flow Analysis Baseline")

with profiler.profile("O2-origin-destination-baseline"):

    #base with trip counts
    o2_pairs = (clean_trips
        .withColumn("year_month", F.date_format(F.col("tpep_pickup_datetime"), "yyyy-MM"))
        .groupBy("PULocationID", "DOLocationID", "year_month")
        .agg(F.count("*").alias("pair_trip_count"))
    )

    zone_lookup_full = (spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(dataFolder + "taxi_zone_lookup.csv")
        .select(
            F.col("LocationID").cast("integer").alias("zone_id"),
            F.col("Borough").alias("borough"),
            F.col("Zone").alias("zone_name")
        )
    )

    zone_pickup = zone_lookup_full.select(
        F.col("zone_id").alias("PULocationID"),
        F.col("borough").alias("pickup_borough"),
        F.col("zone_name").alias("pickup_zone")
    )
    zone_dropoff = zone_lookup_full.select(
        F.col("zone_id").alias("DOLocationID"),
        F.col("borough").alias("dropoff_borough"),
        F.col("zone_name").alias("dropoff_zone")
    )
 
    o2_named = (o2_pairs
        .join(zone_pickup, on="PULocationID", how="inner")
        .join(zone_dropoff, on="DOLocationID", how="inner")
    )

    w_pickup_month = Window.partitionBy("PULocationID", "year_month")
    
    o2_shares = (o2_named
        .withColumn("total_outbound_trips", F.sum("pair_trip_count").over(w_pickup_month)) #each pul month total trips
        .withColumn("route_outbound_share", F.col("pair_trip_count") / F.col("total_outbound_trips"))
    )

    w_borough_rank = Window.partitionBy("pickup_borough", "year_month").orderBy(F.col("pair_trip_count").desc())
    
    o2_top20 = (o2_shares
        .withColumn("borough_route_rank", F.rank().over(w_borough_rank))
        .filter(F.col("borough_route_rank") <= 20)
    )

    o2_quarters = (o2_top20
        .withColumn("quarter", F.quarter(F.to_date(F.col("year_month"), "yyyy-MM"))) 
        .filter(F.col("quarter").isin([1, 3])) 
        .groupBy("PULocationID", "DOLocationID", "pickup_borough", "pickup_zone", "dropoff_zone", "quarter")
        .agg(F.avg("route_outbound_share").alias("avg_quarter_share"))
    )

    o2_q1 = o2_quarters.filter(F.col("quarter") == 1).alias("q1")
    o2_q3 = o2_quarters.filter(F.col("quarter") == 3).alias("q3")

    o2_reversals = (o2_q1
        .join(
            o2_q3,
            (F.col("q1.PULocationID") == F.col("q3.PULocationID")) &
            (F.col("q1.DOLocationID") == F.col("q3.DOLocationID")),
            how="inner"
        )
        .withColumn("share_delta", F.col("q3.avg_quarter_share") - F.col("q1.avg_quarter_share"))
        .select(
            F.col("q1.pickup_borough"),
            F.col("q1.pickup_zone"),
            F.col("q1.dropoff_zone"),
            F.col("q1.avg_quarter_share").alias("q1_share"),
            F.col("q3.avg_quarter_share").alias("q3_share"),
            "share_delta"
        )
        .orderBy(F.abs(F.col("share_delta")).desc()) 
    )

    print("--- O2 Top 10 Route Share Reversals (Q1 vs Q3) ---")
    o2_reversals.show(10, truncate=False)

spark.sparkContext.setJobGroup("", "")

--- O2 Top 10 Route Share Reversals (Q1 vs Q3) ---


+--------------+---------------------------+-----------------------+-------------------+-------------------+--------------------+
|pickup_borough|pickup_zone                |dropoff_zone           |q1_share           |q3_share           |share_delta         |
+--------------+---------------------------+-----------------------+-------------------+-------------------+--------------------+
|Staten Island |West Brighton              |Bloomfield/Emerson Hill|1.0                |0.2                |-0.8                |
|Staten Island |Stapleton                  |Stapleton              |1.0                |0.2222222222222222 |-0.7777777777777778 |
|Staten Island |Westerleigh                |Newark Airport         |1.0                |0.5                |-0.5                |
|Staten Island |South Beach/Dongan Hills   |Red Hook               |0.3214285714285714 |0.5833333333333333 |0.26190476190476186 |
|Staten Island |Bloomfield/Emerson Hill    |Bloomfield/Emerson Hill|1.0                |0.

# O1 Optimization

In [55]:
# # Enable Adaptive Query Execution (AQE)
# # its for the dataskew - splitting heavy partitions and merging small partitions together

spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.localShuffleReader.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", "32")


In [65]:
#optimized with using the broadcast join and AQE
spark.sparkContext.setJobGroup("O1_ANOMALY_OPTIMIZED", "O1: Fare Anomaly Detection Optimized")

with profiler.profile("O1-fare-anomaly-optimized"):
    #creating the base
    o1_base_opt = (clean_trips
        .filter(F.col("trip_distance") > 0)
        .withColumn("fare_per_dist", F.col("fare_amount") / F.col("trip_distance"))
        .withColumn("hour_of_day", F.hour(F.col("tpep_pickup_datetime")))
        .withColumn("is_weekend", F.dayofweek(F.col("tpep_pickup_datetime")).isin([1, 7]))
    )

    o1_stats_opt = (o1_base_opt
        .groupBy("PULocationID", "hour_of_day")
        .agg(
            F.mean("fare_per_dist").alias("mean_fare_per_dist"),
            F.stddev("fare_per_dist").alias("std_fare_per_dist")
        )
    )
    #broadcasts joins 
    o1_flagged_opt = (o1_base_opt
        .join(
            F.broadcast(o1_stats_opt), 
            on=["PULocationID", "hour_of_day"], 
            how="inner"
        )
        .join(
            F.broadcast(
                zones.select(
                    F.col("PULocationID"),
                    "pickup_borough"
                )
            ),
            on="PULocationID",
            how="inner"
        )
        .withColumn(
            "is_fare_anomaly",
            F.when(
                F.col("std_fare_per_dist").isNotNull() & (F.col("std_fare_per_dist") > 0),
                F.col("fare_per_dist") > (F.col("mean_fare_per_dist") + (3 * F.col("std_fare_per_dist")))
            ).otherwise(False)
        )
    )
 
    o1_summary_opt = (o1_flagged_opt 
        .groupBy("pickup_borough", "payment_type", "is_weekend")
        .agg(
            F.count("*").alias("total_trips"),
            F.sum(F.when(F.col("is_fare_anomaly"), 1).otherwise(0)).alias("anomaly_trips"), #true if anomaly
            (F.sum(F.when(F.col("is_fare_anomaly"), 1).otherwise(0)) / F.count("*")).alias("anomaly_rate") 
        )
        .orderBy("pickup_borough", "payment_type", "is_weekend")
    )

    print("--- O1 Fare Anomaly Summary (Optimized) ---")
    o1_summary_opt.show(10, truncate=False)

profiler.compare("O1-fare-anomaly-baseline", "O1-fare-anomaly-optimized")
spark.sparkContext.setJobGroup("", "")

--- O1 Fare Anomaly Summary (Optimized) ---


+--------------+------------+----------+-----------+-------------+--------------------+
|pickup_borough|payment_type|is_weekend|total_trips|anomaly_trips|anomaly_rate        |
+--------------+------------+----------+-----------+-------------+--------------------+
|Bronx         |1           |false     |101567     |1127         |0.011096123740978861|
|Bronx         |1           |true      |23029      |277          |0.0120283121281862  |
|Bronx         |2           |false     |4097       |118          |0.02880156211862338 |
|Bronx         |2           |true      |1826       |44           |0.024096385542168676|
|Bronx         |3           |false     |257        |39           |0.1517509727626459  |
|Bronx         |3           |true      |103        |6            |0.05825242718446602 |
|Bronx         |4           |false     |463        |40           |0.08639308855291576 |
|Bronx         |4           |true      |241        |23           |0.0954356846473029  |
|Brooklyn      |1           |fal

In [60]:
#FAIL WITH WINDOW
#mention in raport that thought that avoiding join with window will benefit but didnot

spark.sparkContext.setJobGroup("O1_ANOMALY_OPTIMIZED", "O1: Fare Anomaly Detection Optimized")

#plan
#broadcast join for zone lookup
#calc on window to avoid the shuffle

with profiler.profile("O1-fare-anomaly-optimized"):
    
    w_bucket = Window.partitionBy("PULocationID", "hour_of_day")

    # 1. Feature extraction + Inline Window Statistics + Anomaly Flagging in a single pass
    o1_flagged_opt = (clean_trips
        .filter(F.col("trip_distance") > 0)
        .withColumn("fare_per_dist", F.col("fare_amount") / F.col("trip_distance"))
        .withColumn("hour_of_day", F.hour(F.col("tpep_pickup_datetime")))
        .withColumn("is_weekend", F.dayofweek(F.col("tpep_pickup_datetime")).isin([1, 7])) #base
        
        #calculating here to avoid shuffle join
        .withColumn("mean_fare_per_dist", F.mean("fare_per_dist").over(w_bucket)) #for each pulid and hour
        .withColumn("std_fare_per_dist", F.stddev("fare_per_dist").over(w_bucket)) 
        
        .withColumn(
            "is_fare_anomaly", #flagging
            F.when(
                F.col("std_fare_per_dist").isNotNull() & (F.col("std_fare_per_dist") > 0), #1st arg
                F.col("fare_per_dist") > (F.col("mean_fare_per_dist") + (3 * F.col("std_fare_per_dist"))) #condit
            ).otherwise(False)
        )

        #broadcasting zonelookup table
        .join(
            F.broadcast(
                zones.select(
                    F.col("PULocationID"),
                    "pickup_borough"
                )
            ),
            on="PULocationID",
            how="inner"
        )
    )

    o1_summary_opt = (o1_flagged_opt
        .groupBy("pickup_borough", "payment_type", "is_weekend")
        .agg(
            F.count("*").alias("total_trips"),
            F.sum(F.when(F.col("is_fare_anomaly"), 1).otherwise(0)).alias("anomaly_trips"), #true if anomaly
            (F.sum(F.when(F.col("is_fare_anomaly"), 1).otherwise(0)) / F.count("*")).alias("anomaly_rate") 
        )
        .orderBy("pickup_borough", "payment_type", "is_weekend")
    )

    print("--- O1 Fare Anomaly Summary (Optimized) ---")
    o1_summary_opt.show(10, truncate=False)

print(profiler.compare("O1-fare-anomaly-baseline", "O1-fare-anomaly-optimized"))
spark.sparkContext.setJobGroup("", "")

--- O1 Fare Anomaly Summary (Optimized) ---


+--------------+------------+----------+-----------+-------------+--------------------+
|pickup_borough|payment_type|is_weekend|total_trips|anomaly_trips|anomaly_rate        |
+--------------+------------+----------+-----------+-------------+--------------------+
|Bronx         |1           |false     |101567     |1127         |0.011096123740978861|
|Bronx         |1           |true      |23029      |277          |0.0120283121281862  |
|Bronx         |2           |false     |4097       |118          |0.02880156211862338 |
|Bronx         |2           |true      |1826       |44           |0.024096385542168676|
|Bronx         |3           |false     |257        |39           |0.1517509727626459  |
|Bronx         |3           |true      |103        |6            |0.05825242718446602 |
|Bronx         |4           |false     |463        |40           |0.08639308855291576 |
|Bronx         |4           |true      |241        |23           |0.0954356846473029  |
|Brooklyn      |1           |fal

# O2 Optimization

In [61]:
spark.sparkContext.setJobGroup("O2_FLOWS_OPTIMIZED", "O2: Origin-Destination Flow Analysis Optimized")

with profiler.profile("O2-origin-destination-optimized"):
    
   # pair with trips count
    o2_pairs_opt = (clean_trips
        .withColumn("year_month", F.date_format(F.col("tpep_pickup_datetime"), "yyyy-MM"))
        .groupBy("PULocationID", "DOLocationID", "year_month")
        .agg(F.count("*").alias("pair_trip_count"))
    )

    #broadcast join only the borough needed for ranking
    zone_borough_bc = F.broadcast(
        zone_lookup_full.select(
            F.col("zone_id").alias("PULocationID"), 
            F.col("borough").alias("pickup_borough")
        )
    )
    
    o2_with_borough = o2_pairs_opt.join(zone_borough_bc, on="PULocationID", how="inner")

    w_pickup_month = Window.partitionBy("PULocationID", "year_month")
    w_borough_rank = Window.partitionBy("pickup_borough", "year_month").orderBy(F.col("pair_trip_count").desc())

    o2_top20_opt = (o2_with_borough #
        .withColumn("total_outbound_trips", F.sum("pair_trip_count").over(w_pickup_month)) #each loc and month total trips
        .withColumn("route_outbound_share", F.col("pair_trip_count") / F.col("total_outbound_trips")) 
        .withColumn("borough_route_rank", F.rank().over(w_borough_rank))
        .filter(F.col("borough_route_rank") <= 20) # early filtering
    )

    o2_reversals_raw = (o2_top20_opt
        .withColumn("quarter", F.quarter(F.to_date(F.col("year_month"), "yyyy-MM")))
        .filter(F.col("quarter").isin([1, 3]))
        .groupBy("PULocationID", "DOLocationID", "pickup_borough")
        .agg(
            F.avg(F.when(F.col("quarter") == 1, F.col("route_outbound_share"))).alias("q1_share"),
            F.avg(F.when(F.col("quarter") == 3, F.col("route_outbound_share"))).alias("q3_share")
        )
        .filter(F.col("q1_share").isNotNull() & F.col("q3_share").isNotNull())
        .withColumn("share_delta", F.col("q3_share") - F.col("q1_share"))
    )

# braodcast join
    zone_p_bc = F.broadcast(
        zone_lookup_full.select(
            F.col("zone_id").alias("PULocationID"),
            F.col("zone_name").alias("pickup_zone")
        )
    )
    zone_d_bc = F.broadcast(
        zone_lookup_full.select(
            F.col("zone_id").alias("DOLocationID"),
            F.col("zone_name").alias("dropoff_zone")
        )
    )

    o2_reversals_opt = (o2_reversals_raw
        .join(zone_p_bc, on="PULocationID", how="inner")
        .join(zone_d_bc, on="DOLocationID", how="inner")
        .select("pickup_borough", "pickup_zone", "dropoff_zone", "q1_share", "q3_share", "share_delta")
        .orderBy(F.abs(F.col("share_delta")).desc())
    )

    print("--- O2 Top 10 Route Share Reversals (Optimized) ---")
    o2_reversals_opt.show(10, truncate=False)

print(profiler.compare("O2-origin-destination-baseline", "O2-origin-destination-optimized"))
spark.sparkContext.setJobGroup("", "")

--- O2 Top 10 Route Share Reversals (Optimized) ---


+--------------+---------------------------+-----------------------+-------------------+-------------------+--------------------+
|pickup_borough|pickup_zone                |dropoff_zone           |q1_share           |q3_share           |share_delta         |
+--------------+---------------------------+-----------------------+-------------------+-------------------+--------------------+
|Staten Island |West Brighton              |Bloomfield/Emerson Hill|1.0                |0.2                |-0.8                |
|Staten Island |Stapleton                  |Stapleton              |1.0                |0.2222222222222222 |-0.7777777777777778 |
|Staten Island |Westerleigh                |Newark Airport         |1.0                |0.5                |-0.5                |
|Staten Island |South Beach/Dongan Hills   |Red Hook               |0.3214285714285714 |0.5833333333333333 |0.26190476190476186 |
|Staten Island |Bloomfield/Emerson Hill    |Bloomfield/Emerson Hill|1.0                |0.

# O3 - Demand Forecasting Features

In [63]:

def test_fun(isBefore: bool):
    daily = clean_trips.withColumn("pickup_date", F.to_date("tpep_pickup_datetime")) \
        .withColumn("hour_of_day", F.hour("tpep_pickup_datetime")) \
        .groupBy("PULocationID", "pickup_date", "hour_of_day") \
        .agg(F.count("*").alias("trip_count")) \
        .withColumn("day_of_week", F.dayofweek("pickup_date"))


    hist = daily \
        .groupBy("PULocationID", "day_of_week", "hour_of_day") \
        .agg(F.avg("trip_count").alias("hist_mean"), F.variance("trip_count").alias("hist_variance"))

    lagged = daily \
        .select(
            "PULocationID",
            F.date_add("pickup_date", 28).alias("pickup_date"),
            "hour_of_day",
            F.col("trip_count").alias("lag_4w_count")   
        )

    # old version with SortMergeJoin
    if isBefore:
        with_lag = daily.join(
            lagged,
            on=["PULocationID", "pickup_date", "hour_of_day"],
            how="left"
        )

        with_stats = with_lag.join(
            hist,
            on=["PULocationID", "day_of_week", "hour_of_day"],
            how="left"
        )

    # new version with BroadcastJoin
    else:
        with_stats = daily \
            .join(
                F.broadcast(lagged),
                on=["PULocationID", "pickup_date", "hour_of_day"],
                how="left"
            ) \
            .join(
                F.broadcast(hist),
                on=["PULocationID", "day_of_week", "hour_of_day"],
                how="left"
        )

    demand = with_stats \
        .withColumn("demand_index", F.when(F.col("hist_mean") > 0, F.col("trip_count") / F.col("hist_mean"))) \
        .withColumn("high_demand", F.coalesce(F.col("demand_index") > 2.0, F.lit(False)))

    zones_o3 = spark.read.csv(zone_lookup_path, header=True) \
        .select(
            F.col("LocationID").cast(IntegerType()).alias("PULocationID"),
            F.col("Borough").alias("pickup_borough"),
            F.col("Zone").alias("pickup_zone"),
            F.col("service_zone")
        )

    # old version with SortMergeJoin
    if isBefore:
        result = demand.join(zones_o3, on="PULocationID", how="left")
        result.write.mode("overwrite") \
            .parquet("/project/output/o3_demand_features_before")
    # new version with BroadcastJoin
    else:
        result = demand.join(
            F.broadcast(zones_o3),
            on="PULocationID", 
            how="left"
        )
        result.write.mode("overwrite") \
            .parquet("/project/output/o3_demand_features_after")

isBefore = True
test_fun(isBefore)
test_fun(not isBefore)

profiler.compare("o3_demand_forecasting_features_before", "o3_demand_forecasting_features_after")

,o3_demand_forecasting_features_before,o3_demand_forecasting_features_after,change
metric,,,
wall_time_s,112.07,96.79,-13.6%
num_jobs,1,3,+200.0%
num_stages,9,9,+0.0%
num_tasks,1031,1031,+0.0%
executor_run_time_ms,147894,116363,-21.3%
executor_cpu_time_ms,107477,78072,-27.4%
input_bytes,31.9MB,31.9MB,+0.0%
output_bytes,7.4MB,10.0MB,+35.2%
shuffle_read_bytes,126.7MB,101.7MB,-19.7%


In [62]:
profiler.report()

,wall_time_s,num_jobs,num_stages,num_tasks,executor_run_time_ms,executor_cpu_time_ms,input_bytes,output_bytes,shuffle_read_bytes,shuffle_write_bytes,memory_spilled_bytes,disk_spilled_bytes
run,,,,,,,,,,,,
0-smoke-test,18.77,2,3,203,10614,7103,21.9KB,0.0B,21.3KB,21.3KB,0.0B,0.0B
task-1_read_data,1.29,1,1,1,379,219,13.2KB,0.0B,0.0B,0.0B,0.0B,0.0B
task-1_clean_data,83.55,1,2,210,28547,19512,142.9KB,0.0B,1.4KB,1.4KB,0.0B,0.0B
task-2_temporal_demand_analysis,189.79,2,3,211,119491,41434,1.6GB,1.2MB,557.2KB,557.2KB,256.0KB,0.0B
task-2_show_10_lines,17.02,3,5,26,31692,23435,1.6GB,0.0B,13.9KB,557.2KB,256.0KB,0.0B
task-3_zone_level_statistics,98.14,3,14,2221,88387,54228,1.5GB,1.1MB,2.3MB,2.1MB,0.0B,0.0B
O1-fare-anomaly-baseline,65.64,1,6,621,119529,105540,21.3MB,0.0B,681.3MB,681.3MB,128.0MB,19.2MB
O2-origin-destination-baseline,120.76,3,19,2624,202713,182414,21.3MB,0.0B,28.6MB,28.6MB,0.0B,0.0B
o3_demand_forecasting_features,117.78,1,9,1031,169215,127576,31.9MB,7.4MB,126.7MB,126.7MB,0.0B,0.0B


In [28]:
result.explain(mode="extended")

== Parsed Logical Plan ==
'Join UsingJoin(LeftOuter, [PULocationID])
:- Project [PULocationID#47, day_of_week#1254, hour_of_day#1237, pickup_date#1236, trip_count#1238L, lag_4w_count#1297L, hist_mean#1255, hist_variance#1256, demand_index#1354, coalesce((demand_index#1354 > 2.0), false) AS high_demand#1355]
:  +- Project [PULocationID#47, day_of_week#1254, hour_of_day#1237, pickup_date#1236, trip_count#1238L, lag_4w_count#1297L, hist_mean#1255, hist_variance#1256, CASE WHEN (hist_mean#1255 > cast(0 as double)) THEN (cast(trip_count#1238L as double) / cast(hist_mean#1255 as double)) END AS demand_index#1354]
:     +- Project [PULocationID#47, day_of_week#1254, hour_of_day#1237, pickup_date#1236, trip_count#1238L, lag_4w_count#1297L, hist_mean#1255, hist_variance#1256]
:        +- Join LeftOuter, (((PULocationID#47 = PULocationID#1320) AND (day_of_week#1254 = day_of_week#1336)) AND (hour_of_day#1237 = hour_of_day#1334))
:           :- Project [PULocationID#47, pickup_date#1236, hour_of_d